# Capítulo 10 · Iteradores

*Traducción y adaptación de [A Whirlwind Tour of Python](https://github.com/jakevdp/WhirlwindTourOfPython)
de Jake VanderPlas (O'Reilly, 2016), publicado con licencia [CC0](https://creativecommons.org/publicdomain/zero/1.0/deed.es).
Adaptación: [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es).*

📖 Lectura complementaria del curso de Python de MiAtlas.cl · se abre en Colab o en JupyterLite, sin GPU.

Con frecuencia, una parte importante del análisis de datos es repetir un cálculo parecido una y otra vez, de forma
automática. Por ejemplo, puede que tengas una tabla de nombres que quieras separar en nombre y apellido, o una de
fechas que quieras convertir a un formato estándar.
Una de las respuestas de Python a esto es la sintaxis de los *iteradores*.
Ya la vimos con `range`:

In [ ]:
for i in range(10):
    print(i, end=' ')

Aquí vamos a profundizar un poco.
Resulta que `range` no es una lista, sino algo que se puede *recorrer* con un iterador, y entender cómo funciona es
clave para comprender una amplia gama de funcionalidades muy útiles de Python.

> 📝 **Nota de MiAtlas.cl:** en rigor, `range` es un *iterable* (un objeto que entrega un iterador cuando se lo
> pides con `iter`), no un iterador. La diferencia se ve más abajo: un iterador se gasta al recorrerlo; un `range`
> se puede recorrer muchas veces.

## Recorrer listas

Los iteradores quizás se entienden mejor en el caso concreto de recorrer una lista.
Considera lo siguiente:

In [ ]:
for valor in [2, 4, 6, 8, 10]:
    # hace alguna operación
    print(valor + 1, end=' ')

La conocida sintaxis `for x in y` permite repetir una operación para cada valor de la lista.
Que el código se parezca tanto a su descripción en lenguaje natural («*para* [cada] valor *en* [la] lista») es una
de las decisiones de sintaxis que hacen de Python un lenguaje tan intuitivo de aprender y de usar.

Pero lo que se ve a simple vista no es lo que *realmente* ocurre.
Cuando escribes algo como `for val in L`, el intérprete de Python comprueba si `L` tiene una interfaz de *iterador*,
algo que tú mismo puedes comprobar con la función incorporada `iter`:

In [ ]:
iter([2, 4, 6, 8, 10])

Es este objeto iterador el que entrega la funcionalidad que necesita el bucle `for`.
El objeto `iter` es un contenedor que te da acceso al siguiente objeto mientras haya uno, como se ve con la función
incorporada `next`:

In [ ]:
I = iter([2, 4, 6, 8, 10])

In [ ]:
print(next(I))

In [ ]:
print(next(I))

In [ ]:
print(next(I))

¿Para qué sirve este nivel de indirección?
Resulta ser increíblemente útil, porque permite que Python trate como listas cosas que *en realidad no son listas*.

> 📝 **Nota de MiAtlas.cl:** cuando un iterador se agota, `next` lanza la excepción `StopIteration`; así sabe el bucle
> `for` que debe terminar. Si quieres evitar la excepción, `next(I, valor_por_omision)` devuelve ese valor al final.

## `range()`: una lista no siempre es una lista

Quizás el ejemplo más común de esta iteración indirecta es la función `range()`, que no devuelve una lista, sino un
objeto especial `range()`:

In [ ]:
range(10)

`range`, igual que una lista, entrega un iterador:

In [ ]:
iter(range(10))

Así que Python sabe tratarlo *como si* fuera una lista:

In [ ]:
for i in range(10):
    print(i, end=' ')

La ventaja de la indirección del iterador es que *la lista completa nunca se crea explícitamente*.
Lo podemos ver con un rango que saturaría la memoria del equipo si de verdad se creara:

In [ ]:
N = 10 ** 12
for i in range(N):
    if i >= 10: break
    print(i, end=', ')

Si `range` creara de verdad esa lista de un billón de valores, ocuparía decenas de terabytes de memoria: un desperdicio,
considerando que ignoramos todo salvo los 10 primeros valores.

De hecho, no hay ninguna razón para que un iterador tenga que terminar.
El módulo `itertools` de Python tiene una función `count` que actúa como un rango infinito:

In [ ]:
from itertools import count

for i in count():
    if i >= 10:
        break
    print(i, end=', ')

Si no hubiéramos puesto un `break` en el bucle, seguiría contando tranquilamente hasta que el proceso se interrumpa o
se detenga a mano (por ejemplo, con `Ctrl-C` en una terminal, o con el botón de detener de Colab o de Jupyter).

## Iteradores útiles

Esta sintaxis de iteradores se usa de forma casi universal en los tipos incorporados de Python y en los objetos más
propios de la ciencia de datos que exploraremos en capítulos posteriores.
Aquí veremos algunos de los iteradores más útiles del lenguaje:

### `enumerate`

A menudo necesitas recorrer no solo los valores de una lista, sino también llevar la cuenta del índice.
Podrías sentir la tentación de hacerlo así:

In [ ]:
L = [2, 4, 6, 8, 10]
for i in range(len(L)):
    print(i, L[i])

Aunque funciona, Python ofrece una sintaxis más limpia con el iterador `enumerate`:

In [ ]:
for i, val in enumerate(L):
    print(i, val)

Esta es la forma más «pythónica» de enumerar los índices y los valores de una lista.

> 📝 **Nota de MiAtlas.cl:** `enumerate` acepta un argumento `start` para empezar a contar desde otro número, por
> ejemplo `enumerate(L, start=1)`.

### `zip`

Otras veces tendrás varias listas que quieres recorrer al mismo tiempo.
Podrías recorrer el índice como en el ejemplo poco pythónico de antes, pero es mejor usar el iterador `zip`, que une
(como un cierre de cremallera) varios iterables:

In [ ]:
L = [2, 4, 6, 8, 10]
R = [3, 6, 9, 12, 15]
for lval, rval in zip(L, R):
    print(lval, rval)

Se puede unir cualquier cantidad de iterables y, si tienen largos distintos, el más corto determina el largo del
`zip`.

> 📝 **Nota de MiAtlas.cl:** desde Python 3.10, `zip(L, R, strict=True)` lanza un `ValueError` si los iterables no
> tienen el mismo largo, en lugar de cortar en silencio. Es útil cuando un largo distinto indica un error en los datos.

### `map` y `filter`

El iterador `map` toma una función y la aplica a los valores de un iterador:

In [ ]:
# los 10 primeros cuadrados
cuadrado = lambda x: x ** 2
for val in map(cuadrado, range(10)):
    print(val, end=' ')

El iterador `filter` se parece, salvo que solo deja pasar los valores para los que la función de filtro resulta
`True`:

In [ ]:
# los valores hasta 10 para los que x % 2 es cero
es_par = lambda x: x % 2 == 0
for val in filter(es_par, range(10)):
    print(val, end=' ')

Las funciones `map` y `filter`, junto con la función `reduce` (que vive en el módulo `functools` de Python), son
componentes fundamentales del estilo de *programación funcional*, que, aunque no es el estilo dominante en el mundo
de Python, tiene sus defensores entusiastas (mira, por ejemplo, la biblioteca
[toolz](https://toolz.readthedocs.io/en/latest/)).

### Iteradores como argumentos de funciones

En el capítulo 8, *Funciones*, vimos que `*args` y `**kwargs` sirven para pasar secuencias y diccionarios a las
funciones. Resulta que la sintaxis `*args` funciona no solo con secuencias, sino con cualquier iterador:

In [ ]:
print(*range(10))

Así, por ejemplo, podemos ponernos ingeniosos y comprimir el ejemplo de `map` de antes en esto:

In [ ]:
print(*map(lambda x: x ** 2, range(10)))

Este truco permite responder la eterna pregunta de los foros de quienes aprenden Python: ¿por qué no existe una
función `unzip()` que haga lo contrario de `zip()`?
Si te encierras en un armario oscuro y lo piensas un rato, quizás te des cuenta de que lo contrario de `zip()` es…
¡`zip()`! La clave es que `zip()` puede unir cualquier cantidad de iteradores o secuencias. Observa:

In [ ]:
L1 = (1, 2, 3, 4)
L2 = ('a', 'b', 'c', 'd')

In [ ]:
z = zip(L1, L2)
print(*z)

In [ ]:
z = zip(L1, L2)
nueva_L1, nueva_L2 = zip(*z)
print(nueva_L1, nueva_L2)

Medítalo un rato. Si entiendes por qué funciona, ¡habrás avanzado mucho en la comprensión de los iteradores de Python!

> 📝 **Nota de MiAtlas.cl:** fíjate en que creamos `z` de nuevo antes de la segunda celda. Un `zip` es un iterador:
> después de recorrerlo con `print(*z)` queda agotado, y recorrerlo otra vez no entregaría nada.

## Iteradores especializados: `itertools`

Ya vimos brevemente el rango infinito `itertools.count`.
El módulo `itertools` contiene muchísimos iteradores útiles; vale la pena explorarlo para ver qué ofrece.
Como ejemplo, considera la función `itertools.permutations`, que recorre todas las permutaciones de una secuencia:

In [ ]:
from itertools import permutations
p = permutations(range(3))
print(*p)

De forma parecida, la función `itertools.combinations` recorre todas las combinaciones únicas de `N` valores de una
lista:

In [ ]:
from itertools import combinations
c = combinations(range(4), 2)
print(*c)

Algo relacionado es el iterador `product`, que recorre todos los pares (o tuplas) posibles entre dos o más iterables:

In [ ]:
from itertools import product
p = product('ab', range(3))
print(*p)

Hay muchos más iteradores útiles en `itertools`: la lista completa, con ejemplos, está en la
[documentación de Python](https://docs.python.org/es/3/library/itertools.html).

> 📝 **Nota de MiAtlas.cl:** dos agregados recientes de `itertools`: `pairwise` (Python 3.10) recorre pares de
> elementos consecutivos, y `batched` (Python 3.12) agrupa en lotes de un tamaño fijo:

In [ ]:
from itertools import pairwise, batched
print(*pairwise([1, 2, 3, 4]))
print(*batched(range(7), 3))

## Pruébalo tú

Tienes una lista de nombres. Usa `enumerate` (con `start=1`) para construir una **lista de tuplas** `(número, nombre)`
que empiece en 1, por ejemplo `[(1, 'Ana'), (2, 'Bruno'), …]`, y guárdala en `resultado`.

In [ ]:
nombres = ['Ana', 'Bruno', 'Carla', 'Diego']

resultado = None

# Escribe aquí tu código

In [ ]:
# Comprobación
if resultado is None:
    print("Aún no lo resuelves.")
elif resultado == [(1, 'Ana'), (2, 'Bruno'), (3, 'Carla'), (4, 'Diego')]:
    print("✅ ¡Correcto!")
else:
    print("❌ Revisa tu respuesta")

**Sigue en MiAtlas.cl:** tema «Iteradores y generadores».

**Original:** [10-Iterators.ipynb](https://github.com/jakevdp/WhirlwindTourOfPython/blob/master/10-Iterators.ipynb)